In [2]:
import sys
import os
from tqdm import tqdm
sys.path.append("C:\\Users\\btenhaaf\\Documents\\Packages\\FermSystem\\")

from functools import partial
from importlib import reload
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math,Markdown

In [3]:
## Custom functions
import FockSystem.FockSystem as fst

In [4]:
from itertools import product,combinations,permutations


In [5]:
def print_md(item):
    display(Markdown(item._repr_markdown_()))

In [6]:
possible_operators = [fst.OperSequence(('cr',0,'up')).oper_list[0][0],fst.OperSequence(('cr',0,'dwn')).oper_list[0][0], fst.OperSequence(('an',0,'up')).oper_list[0][0], fst.OperSequence(('an',0,'dwn')).oper_list[0][0]]

def get_combinations(lst, n):
    return list(combinations(lst, n))

In [7]:
fs = fst.FockSystemBase()
def generate_combinations(pos_ops, choose_n):
    all_options = []
    for i in range(1,len(possible_operators)+1):
        combs = get_combinations(pos_ops, i)
        all_options.extend([fs.normal_order_naive(list(c))[0] for c in combs])

    all_oper_lists = []
    combs = get_combinations(all_options, choose_n)
    all_oper_lists.extend([list(c) for c in combs])
    return all_oper_lists

In [8]:
def check_all_unit_powers(pos_ops, choose_n, exponent, possible_signs = [1, -1, 1j, -1j]):
    all_lists = generate_combinations(pos_ops,choose_n)
    
    success_list = []

    possible_weights = np.array(list(product(possible_signs, repeat=choose_n)))
    print(f"Trying {len(all_lists)*len(possible_weights)} possible combinations")
    for oper_list in tqdm(all_lists):
        for pos_w in possible_weights:
            weights = list(pos_w)
            operator = fst.OperSequence(oper_list, weights=weights,bypass_parse=True)
            result = operator**exponent
            if result.oper_list:
                if len(result.oper_list)==1:
                    if isinstance(result.oper_list[0],int):
                        if result.weights[0]*result.oper_list[0] == 1.0:
                                success_list.append(operator)
    print(f"Found {len(success_list)} possible operator sequences with oper^{exponent} = 1 (considered {choose_n} terms)")
    return success_list

In [9]:
def check_all_unit_powers_numpified(pos_ops, choose_n, exponent, possible_signs = [1,-1,1j,-1j]):
    all_lists = generate_combinations(pos_ops,choose_n)
    possible_weights = np.array(list(product(possible_signs, repeat=choose_n)))

    number_of_options = len(all_lists)*len(possible_weights)
    print(f"Trying {number_of_options} possible combinations")

    generated_operators = np.empty(number_of_options, dtype=object)
    i=0
    for oper_list in tqdm(all_lists):
        for pos_w in possible_weights:
            weights = list(pos_w)
            generated_operators[i] = fst.OperSequence(oper_list,weights=weights, bypass_parse=True)
            i+=1

    identity_op = fst.OperSequence([1], bypass_parse=True)
    filt_equal = np.where(generated_operators**exponent == identity_op)
    result = generated_operators[filt_equal]
    print(f"Found {len(result)} possible operator sequences with oper^{exponent} = 1 (considered {choose_n} terms)")

    return result

In [10]:
from time import time

## Example: The Kitaev chain

In [11]:
def doublet_pairs(oper_sequences, conjugation=False):
    indices = range(len(oper_sequences))
    permutations_list = list(permutations(indices, 2))
    result = []
    lengths = []
    for i,j in tqdm(permutations_list):
        H = ~oper_sequences[i]*(oper_sequences[j] >>1) + ~oper_sequences[j]*(oper_sequences[i]>>1) # + ~oper_sequences[i]*oper_sequences[j]
    
        if H % 2:
            result.append((i,j))
            lengths.append(len(H.oper_list))
    print(f"Found {len(result)} possible pairs")
    return result,lengths



In [12]:
def trio_pairs(oper_sequences, conjugation=False):
    indices = range(len(oper_sequences))
    permutations_list = list(permutations(indices, 3))
    lengths = []
    result = []
    for i,j,k in tqdm(permutations_list):
        op1 = oper_sequences[i]
        op2 = oper_sequences[j]
        op3 = oper_sequences[k]

        H = op1*((~op2)>>1) + op2*((~op3)>>1) + op3*((~op1)>>1) + op1*op2 + op2*op3 + op1*op3 + ((op1*op2)>>1) + ((op2*op3)>>1) + ((op1*op3)>>1)
        
    
        if H % 2:
            result.append((i,j,k))
            lengths.append(len(H.oper_list))
    print(f"Found {len(result)} possible pairs")
    return result,lengths



In [13]:
all_operators = []
operators_squared = check_all_unit_powers(possible_operators,choose_n = 2,exponent= 2)

Trying 1680 possible combinations


100%|███████████████████████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 375.74it/s]

Found 8 possible operator sequences with oper^2 = 1 (considered 2 terms)


In [15]:
for op in operators_squared:
    print_md(op)

 $c^{†}_{0,↑}$ $+$ $c_{0,↑}$

 $-$ $c^{†}_{0,↑}$ $-$ $c_{0,↑}$

 j$c^{†}_{0,↑}$ $-$ j$c_{0,↑}$

 $-$ j$c^{†}_{0,↑}$ $+$ j$c_{0,↑}$

 $c^{†}_{0,↓}$ $+$ $c_{0,↓}$

 $-$ $c^{†}_{0,↓}$ $-$ $c_{0,↓}$

 j$c^{†}_{0,↓}$ $-$ j$c_{0,↓}$

 $-$ j$c^{†}_{0,↓}$ $+$ j$c_{0,↓}$

In [16]:
operators_squared = check_all_unit_powers(possible_operators,choose_n = 3,exponent= 2)
all_operators.extend(operators_squared)

Trying 29120 possible combinations


100%|███████████████████████████████████████████████████████████████████████████████| 455/455 [00:04<00:00, 111.12it/s]

Found 64 possible operator sequences with oper^2 = 1 (considered 3 terms)


In [17]:
for op in operators_squared:
    print_md(op)

 $c^{†}_{0,↑}$ $+$ $c^{†}_{0,↓}$ $+$ $c_{0,↑}$

 $c^{†}_{0,↑}$ $-$ $c^{†}_{0,↓}$ $+$ $c_{0,↑}$

 $c^{†}_{0,↑}$ $+$ j$c^{†}_{0,↓}$ $+$ $c_{0,↑}$

 $c^{†}_{0,↑}$ $-$ j$c^{†}_{0,↓}$ $+$ $c_{0,↑}$

 $-$ $c^{†}_{0,↑}$ $+$ $c^{†}_{0,↓}$ $-$ $c_{0,↑}$

...

 j$c^{†}_{0,↓}$ $-$ j$c_{0,↑}$ $-$ j$c_{0,↓}$

 $-$ j$c^{†}_{0,↓}$ $+$ $c_{0,↑}$ $+$ j$c_{0,↓}$

 $-$ j$c^{†}_{0,↓}$ $-$ $c_{0,↑}$ $+$ j$c_{0,↓}$

 $-$ j$c^{†}_{0,↓}$ $+$ j$c_{0,↑}$ $+$ j$c_{0,↓}$

 $-$ j$c^{†}_{0,↓}$ $-$ j$c_{0,↑}$ $+$ j$c_{0,↓}$

In [139]:
all_operators = []
operators_squared = check_all_unit_powers(possible_operators,choose_n = 2,exponent= 2)
all_operators.extend(operators_squared)
operators_squared = check_all_unit_powers(possible_operators,choose_n = 3,exponent= 2)
all_operators.extend(operators_squared)
operators_squared = check_all_unit_powers(possible_operators,choose_n = 4,exponent= 2)
all_operators.extend(operators_squared)

Trying 1680 possible combinations


100%|████████████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 766.80it/s]


Found 8 possible operator sequences with oper^2 = 1 (considered 2 terms)
Trying 29120 possible combinations


100%|████████████████████████████████████████████████████████████████████| 455/455 [00:04<00:00, 109.46it/s]


Found 64 possible operator sequences with oper^2 = 1 (considered 3 terms)
Trying 349440 possible combinations


100%|███████████████████████████████████████████████████████████████████| 1365/1365 [01:13<00:00, 18.59it/s]

Found 88 possible operator sequences with oper^2 = 1 (considered 4 terms)


In [203]:
operators_squared = check_all_unit_powers(possible_operators,choose_n = 5,exponent= 2)
all_operators.extend(operators_squared)

Trying 3075072 possible combinations


100%|███████████████████████████████████████████████████████████████████| 3003/3003 [15:19<00:00,  3.26it/s]

Found 336 possible operator sequences with oper^2 = 1 (considered 5 terms)


In [146]:
hermitian_operators = []
for op in all_ops:
    if op.is_hermitian():
        hermitian_operators.append(op)

In [201]:
commuting_pairs = []
for idx1,op1 in enumerate(hermitian_operators):
    for idx2,op2 in enumerate(hermitian_operators):
        anticom = op1*op2 + op2*op1
        anticom.remove_zero_weight()
        if not (anticom == empty_seq):
            continue
        anticom = op1*(op2>>1) + (op2>>1)*op1
        anticom.remove_zero_weight()
        if not (anticom == empty_seq):
            continue
        commuting_pairs.append((idx1,idx2))

In [159]:
op1,op2 = hermitian_operators[commuting_pairs[100][0]],hermitian_operators[commuting_pairs[100][1]]

In [171]:
hermitian_operators[0]*hermitian_operators[2]

 $-$ 2.0j$c^{†}_{0,↑}$$c_{0,↑}$ $+$ 1j

In [170]:
hermitian_operators[2]

 j$c^{†}_{0,↑}$ $-$ j$c_{0,↑}$

In [186]:
len(commuting_pairs)

304

In [188]:
idx=250
op1,op2 = hermitian_operators[commuting_pairs[idx][0]],hermitian_operators[commuting_pairs[idx][1]]


In [189]:
op1

 j$c^{†}_{0,↓}$$c^{†}_{0,↑}$ $+$ j$c^{†}_{0,↑}$$c_{0,↓}$ $-$ j$c^{†}_{0,↓}$$c_{0,↑}$ $+$ j$c_{0,↓}$$c_{0,↑}$

In [187]:
for idx in range(300,304):
    op1,op2 = hermitian_operators[commuting_pairs[idx][0]],hermitian_operators[commuting_pairs[idx][1]]
    print_md(op1*op2)

 $-$ 2.0j$c^{†}_{0,↑}$$c_{0,↑}$ $+$ 1j

 $-$ 2.0j$c^{†}_{0,↓}$$c_{0,↓}$ $+$ 1j

 2.0j$c^{†}_{0,↓}$$c_{0,↓}$ $+$ -1j

 2.0j$c^{†}_{0,↑}$$c_{0,↑}$ $+$ -1j

In [149]:
for op in hermitian_operators:
    op.symbolic()

$c^{†}_{0,↑}$ + $c_{0,↑}$

$c^{†}_{0,↑}$ + $c_{0,↑}$

$c^{†}_{0,↑}$ + $c_{0,↑}$

$c^{†}_{0,↑}$ + $c_{0,↑}$

$c^{†}_{0,↓}$ + $c_{0,↓}$

...

$c^{†}_{0,↓}$$c^{†}_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↓}$ + $c^{†}_{0,↓}$$c_{0,↑}$ + $c_{0,↓}$$c_{0,↑}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↓}$ + $c^{†}_{0,↓}$$c_{0,↑}$ + $c_{0,↓}$$c_{0,↑}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↓}$ + $c^{†}_{0,↓}$$c_{0,↑}$ + $c_{0,↓}$$c_{0,↑}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↓}$ + $c^{†}_{0,↓}$$c_{0,↑}$ + $c_{0,↓}$$c_{0,↑}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↓}$ + $c^{†}_{0,↓}$$c_{0,↑}$ + $c_{0,↓}$$c_{0,↑}$

In [133]:
for idx in range(50,70):
    i,j = potential_pairs[0]
    op1 = hermitian_ops[i]
    op2 = hermitian_ops[j]
    
    res = op1*op2 + op2*op1
    res.remove_zero_weight()
    res.symbolic()


$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↑}$ + $μ_{0}$$c^{†}_{0,↓}$$c_{0,↓}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↑}$ + $μ_{0}$$c^{†}_{0,↓}$$c_{0,↓}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↑}$ + $μ_{0}$$c^{†}_{0,↓}$$c_{0,↓}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↑}$ + $μ_{0}$$c^{†}_{0,↓}$$c_{0,↓}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↑}$ + $μ_{0}$$c^{†}_{0,↓}$$c_{0,↓}$

...

$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↑}$ + $μ_{0}$$c^{†}_{0,↓}$$c_{0,↓}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↑}$ + $μ_{0}$$c^{†}_{0,↓}$$c_{0,↓}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↑}$ + $μ_{0}$$c^{†}_{0,↓}$$c_{0,↓}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↑}$ + $μ_{0}$$c^{†}_{0,↓}$$c_{0,↓}$

$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ + $c^{†}_{0,↑}$$c_{0,↑}$ + $μ_{0}$$c^{†}_{0,↓}$$c_{0,↓}$

In [131]:
res

 $-$ 8.0$c^{†}_{0,↓}$$c^{†}_{0,↑}$$c_{0,↓}$$c_{0,↑}$ $-$ 4.0$c^{†}_{0,↑}$$c_{0,↑}$ $-$ 4.0$c^{†}_{0,↓}$$c_{0,↓}$ $+$ (2+0j)

In [ ]:

potential_pairs = doublet_pairs(hermitian_ops)[0]
for pair in potential_pairs:
    i = pair[0]
    j = pair[1]
    oper_1 = operators_squared[i]
    oper_2 = operators_squared[j]
    H = (3+5)*oper_1*(oper_2>>1) + (3-5)*oper_2*(oper_1>>1) + oper_1*oper_2 + (oper_1>>1)*(oper_2>>1)
    print_md(H.remove_zero_weight())


## Cubic operators

In [ ]:
operators_cubed = check_all_unit_powers(possible_operators,choose_n = 5,exponent= 3)

In [ ]:
operators_cubed[2]

In [ ]:
op1 = operators_cubed[192]

In [ ]:
op2 = operators_cubed[208]

In [ ]:
options,lengths = doublet_pairs(operators_cubed[192:])

In [ ]:
lengths

In [ ]:
lengths[np.argmin(lengths)]

In [ ]:
cool_options,lengths = trio_pairs(operators_cubed[192:250])

In [ ]:
cool_options[np.argmin(lengths)]

In [ ]:
length = []
for pair in options:
    op1 = operators_cubed[pair[0]]
    op2 = operators_cubed[pair[1]]
    H = (5-3)*op1*((~op2)>>1) + (5+3)*op2*((~op1)>>1)
    length.append(len(H.oper_list))
    

In [ ]:
np.where(options == 25)

In [ ]:
options[np.argmin(length)]

In [ ]:
H = op1*((~op2)>>1) + op2*((~op3)>>1) + op3*((~op1)>>1) + op1*op2 + op2*op3 + op1*op3 + ((op1*op2)>>1) + ((op2*op3)>>1) + ((op1*op3)>>1)
H

In [ ]:
test = fst.OperSequence([[0,1],[2,3],[3,4],[4,5]], bypass_parse=True)

In [ ]:
op1 = operators_cubed[192]
op2 = operators_cubed[193]
op3 = operators_cubed[194]

In [ ]:
H = (7-3)*op1*(op2>>1) + (7+3)*op2*(op1>>1)

In [ ]:
c_dwn = fst.OperSequence(0)
a_dwn = fst.OperSequence(1)
c_up = fst.OperSequence(2)
a_up = fst.OperSequence(3)
MU_up = c_up*a_up
MU_up += (MU_up >>1) 
MU_down = c_dwn*a_dwn
MU_down += (MU_down >> 1) 


In [ ]:
mu_range = np.linspace(-100,100,100)
t_range = np.linspace(-100,100,100)
H[sub2] = 40
## Loop over range and get array, pass to linalg for eigenvalues
result = tu.phase_diagram(H,odd_basis,even_basis, sub1, t_range, t , mu_range)

fig,ax = plt.subplots(ncols=1, figsize = (3,3))
result['E'].plot()